![Week 5 Day 3](../assets/banners/w5d3.png)

# Week 5 Day 3 - Report Cards, Storage and "Chat With Results"

Yesterday GradeWise learned to grade a whole script: one fresh grading run per question with `Send`, guardrails on every mark, and a prompt injection caught red-handed. At the end of `aggregate`, out came a `ReportCard`: a total, a grade band and fifteen `GradeResult`s.

That's a perfectly good data structure. It is also completely useless to Sam. 😄

Today we turn marks into something people use. For **Sam**: a report card with his scores, feedback he can act on and the course-note citations behind every mark, as a web page and a PDF. For **Emma**: every report stored in SQLite and indexed in Chroma, and an agent she can simply **ask**: *"Which question did the class struggle with most?"*

![Week 5 roadmap](../assets/roadmap/week5.png)

### 🎯 By the end of this lab you will
1. 📋 Know the anatomy of a `ReportCard`, and what makes feedback useful
2. ✍️ Write strengths and improvements, free with rules or with an LLM
3. 🌐 Render report cards as HTML (inline, right here) and as an A4 PDF, safely escaped
4. 💾 Store reports in SQLite + a Chroma `reports_cs_midterm` collection
5. 💬 Build and question the "chat with results" agent

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 3 - Writing the Report Card: Scores, Feedback, Citations (HTML/PDF) | 🧪 |
| 2 | Day 3 - Storing Reports and the "Chat With Results" Agent | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Today's marks come from the 🎭 <b>stand-in grader</b> (<code>KeywordGraderModel</code>), so the whole class can be graded for free in a second or two. The marks themselves mean little; the <b>report cards, storage and Q&A agent are the real thing</b>. Everything is written to a sandbox, <code>output/week5/day3/</code>, so it never mixes with your real results. The 🔑 cells use your real models.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT, settings
from gradewise.console import success
from gradewise.evals.compat import import_pandas
from gradewise.ingestion import ingest
from gradewise.rag import get_retriever
from gradewise.schemas import ExamPaper, ReportCard

pd = import_pandas()  # 💡 = `import pandas as pd`, with a fix for locked-down Windows machines

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
FULL_EXAM = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTIONS = {q.id: q for q in FULL_EXAM.questions}  # every question, for the one-answer demos
TRUTH = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))

# 📦 What the batch runs grade: all 15 parts and 10 students, or Q1–Q3 and 2 students with DATA_SIZE=small
exam = datasets.subset_exam(FULL_EXAM)
STUDENTS = datasets.synthetic_students()

# 🆓 A keyword-only (BM25) copy of the CS201 notes: fake embeddings, real keyword search, zero cost
OFFLINE_DB = OUTPUT_DIR / "week5_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
ingest(
    collection="cs201_offline",
    embeddings=fake_embeddings,
    persist_dir=OFFLINE_DB,
    quiet=True,
    files=datasets.kb_files(),
)  # 📦 all 4 PDFs, or unit1 + the handbook with DATA_SIZE=small
bm25 = get_retriever("cs201_offline", k=4, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

success(
    "Ready",
    {
        "Exam": f"{exam.title} ({len(exam.questions)} questions, {exam.total_marks:g} marks)",
        "Students": f"{len(STUDENTS)} synthetic scripts",
        "Offline KB": f"cs201_offline (BM25, {len(datasets.kb_files())} PDFs)",
        "Grading model": settings.grading_model,
    },
)

In [ ]:
import html

from IPython.display import HTML

from gradewise.reports import render_html


def show_report(report: ReportCard, height: int = 760, paper: ExamPaper | None = None) -> None:
    """Render a report card inline. 💡 Inside an <iframe>, so its CSS can't restyle this notebook."""
    page = render_html(report, paper or exam)
    frame = f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:{height}px;border:0"></iframe>'
    display(HTML(f"<div>{frame}</div>"))

## Part 1: Grade the class (rehearsal)

Build the grading graph once with the stand-in, then grade the class with `grade_sheet()`: all ten synthetic students, or two with `DATA_SIZE=small` (`STUDENTS`, from the data cell). This is exactly Day 2's graph, just run once per student.

In [ ]:
from gradewise.grading import build_grading_graph, grade_sheet
from gradewise.grading.fakes import KeywordGraderModel
from gradewise.orchestrator import sheets_from_transcripts

graph = build_grading_graph(KeywordGraderModel(), bm25)  # 🎭 stand-in grader
sheets = sheets_from_transcripts(exam, students=STUDENTS)  # 📦 10 students, or 2 with DATA_SIZE=small
reports = [grade_sheet(exam, sheet, graph=graph) for sheet in sheets]
by_id = {r.student_id: r for r in reports}

pd.DataFrame(
    [
        {
            "student": r.student_id,
            "total": r.total,
            "%": r.percentage,
            "band": r.grade_band,
            "flagged": ", ".join(g.question_id for g in r.results if g.needs_review) or "-",
        }
        for r in reports
    ]
)

## Part 2: The anatomy of a report card

`ReportCard` lives in `gradewise/schemas/report.py`. Build one with `ReportCard.from_results()` and it works out the total, the percentage, the grade band and whether anything still needs review, so these numbers can never disagree with the marks underneath.

In [ ]:
from gradewise.schemas.report import GRADE_BANDS

sam_report = by_id["student_05"]
r = sam_report
print(f"{r.student_id}: {r.total:g}/{r.max_total:g} = {r.percentage:g}% → band {r.grade_band}")
print(f"grade bands (min %, band): {GRADE_BANDS}")
print(
    f"needs_review: {sam_report.needs_review} · results: {len(sam_report.results)} · strengths: {sam_report.strengths}"
)

`strengths` and `improvements` are empty: grading decides marks, not advice. Good feedback is its own job, and it has its own rules:

| Good feedback is… | Not… |
|---|---|
| **specific**: names the question and the topic ("Q7a: subtract the network and broadcast addresses") | "Revise networking." |
| **actionable**: says what to do next | "Could be better." |
| **honest and kind**: second person, warm, true | Praise for a question Sam didn't answer |
| **grounded**: only uses the marks and the examiner's notes | Invented marks or topics |

## Part 3: Writing the feedback

Two writers, same output. `rule_based_feedback()` is free and deterministic: strengths from the questions scoring 75% or more, improvements from those under half, using the **missed criteria** to say what to focus on. A little robotic, but never wrong, and it's the fallback whenever an LLM call fails.

In [ ]:
from gradewise.reports import rule_based_feedback, write_feedback

ruled = rule_based_feedback(sam_report, exam)
print("✅ Strengths")
print("\n".join(f"   • {s}" for s in ruled.strengths))
print("🎯 To improve")
print("\n".join(f"   • {s}" for s in ruled.improvements))

🔑 `write_feedback()` asks the everyday chat model (`gpt-6-luna` in `MODEL_MODE=cheap`, at its normal reasoning effort: the hard judgement, the marks, is already done) for a `ReportFeedback` with structured output. Its prompt includes a compact summary of every mark and missed criterion, wrapped in `<results>` tags: data, not instructions.

In [ ]:
from gradewise.errors import GradeWiseError

try:
    written = write_feedback(sam_report, exam)
    print("✅ Strengths")
    print("\n".join(f"   • {s}" for s in written.strengths))
    print("🎯 To improve")
    print("\n".join(f"   • {s}" for s in written.improvements))
except GradeWiseError as err:
    err.show()
    written = ruled  # 💡 carry on with the rule-based version

## Part 4: The HTML report card

`render_html()` returns **one self-contained page**: inline CSS, no external files, so you can email it, print it or serve it. Here is Sam's, rendered right inside the notebook. Expand a question's *Criteria* to see each mark, its justification and its 📖 citation.

In [ ]:
show_report(written)

And student 10, whose Q8 contained the prompt injection. Look for the amber ⚑ banner at the top and on Q8: the report says, honestly, that a teacher still has to check it. (Student 10 and Q8 aren't in the `DATA_SIZE=small` slice, so the next cell grades that one script on the full paper with the free stand-in.)

In [ ]:
ten = by_id.get("student_10") or grade_sheet(
    FULL_EXAM, sheets_from_transcripts(FULL_EXAM, students=["student_10"])[0], graph=graph
)  # 🎭 stand-in, free
show_report(rule_based_feedback(ten, FULL_EXAM), height=420, paper=FULL_EXAM)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>Every piece of student text is HTML-escaped.</b> A report card shows the student's own words (in justifications) to a teacher, in a browser. If a student writes <code>&lt;script&gt;</code> in an answer and we paste it into HTML unescaped, it runs in Emma's browser with her session. That's layer 5 of Day 2's defence in depth, and the next cell proves it.</span>
        </td>
    </tr>
</table>

In [ ]:
hostile = sam_report.model_copy(
    update={"strengths": ['<script>alert("pwned")</script> nice answer <img src=x onerror=alert(1)>']}
)
page = render_html(hostile, exam)
print("raw <script> in the page?  ", '<script>alert("pwned")' in page)
print("escaped version instead:   ", page[page.index("&lt;script") :][:60])

## Part 5: The PDF report card

Schools still print things. `render_pdf()` uses **PyMuPDF**'s `Story` engine to flow the same content across A4 pages, then draws the navy header band, the score and page numbers with plain drawing calls (the comments in `gradewise/reports/writer.py` explain why: Story draws CSS background colours badly). Let's render page 1 to an image to see it:

In [ ]:
import pymupdf
from IPython.display import Image

from gradewise.reports import render_pdf

SANDBOX = OUTPUT_DIR / "week5" / "day3"
pdf_path = render_pdf(written, exam, SANDBOX / "reports" / "student_05.pdf")
with pymupdf.open(pdf_path) as doc:
    print(f"{pdf_path.relative_to(REPO_ROOT)}: {doc.page_count} page(s)")
    png = doc[0].get_pixmap(dpi=70).tobytes("png")
display(Image(data=png))

`save_report_files()` writes both (`<student>.html` and `<student>.pdf`) in one call; it's what the orchestrator's `write_report` node uses on Day 4.

In [ ]:
from gradewise.reports import save_report_files

paths = save_report_files(written, exam, SANDBOX / "reports")
print({kind: str(path.relative_to(REPO_ROOT)) for kind, path in paths.items()})

## Part 6: Storing reports: SQLite + Chroma

A report card has two lives after it's written, and each needs a different store:

| Store | Holds | Answers questions like | Code |
|---|---|---|---|
| **SQLite** (`gradewise/db.py`) | the full `ReportCard` as JSON, one row per student | "What did student_04 get?" (exact) | `save_report()`, `get_report()`, `list_reports()` |
| **Chroma** (`reports_cs_midterm`) | one chunk per (student, question): marks, justifications, feedback | "Who struggled with subnetting?" (semantic) | `index_report()`, `search_reports()` |

`store_report()` does both. That's week 2's knowledge-base machinery, now pointed at **GradeWise's own output**. Here's one chunk:

In [ ]:
from gradewise.reports import report_documents

docs = report_documents(sam_report, exam)
q7a_doc = next((d for d in docs if d.metadata["question_id"] == "Q7a"), docs[-1])  # 📦 small data: Q3 instead
print(f"id: {q7a_doc.id}\nmetadata: {q7a_doc.metadata}\n\n{q7a_doc.page_content}")

💡 The chunk ID is stable (`cs_midterm:student_05:Q7a`) and Chroma *upserts* by ID, so re-grading a student overwrites their chunks instead of duplicating them. Let's store the whole class in the sandbox, with our free fake embeddings:

In [ ]:
from gradewise.db import get_report, list_reports
from gradewise.reports import search_reports, store_report

STORE = {"db": SANDBOX / "gradewise.db", "embeddings": fake_embeddings, "persist_dir": SANDBOX / "vector_db"}
for report in reports:
    store_report(rule_based_feedback(report, exam), exam, **STORE)

last = reports[-1].student_id  # student_10 (or Sam, with DATA_SIZE=small)
print(
    f"SQLite: {len(list_reports(exam.exam_id, db=STORE['db']))} reports · "
    f"{last} total from the DB: {get_report(exam.exam_id, last, db=STORE['db']).total:g}"
)
hits = search_reports(exam.exam_id, "anything", student_id=last, k=3, **{k: v for k, v in STORE.items() if k != "db"})
print(f"filtered search, {last} only:", [d.id for d in hits])

⚠️ With fake embeddings, *which* chunks come back is random: they carry no meaning. The **filters** are exact, though: ask for `student_10` and you only ever get student 10. With real embeddings (the 🔑 cell in Part 7), the search becomes semantic.

## Part 7: The "chat with results" agent

`gradewise/reports/qa_agent.py` builds a `create_agent` with four tools, bound to one exam:

| Tool | For questions like | Where the numbers come from |
|---|---|---|
| `class_statistics(question_id="")` | "Which question was hardest?" | pandas over SQLite: **exact** |
| `get_student_report(student_id)` | "Why did student_05 lose marks on Q7a?" | SQLite |
| `search_reports(query, student_id, question_id)` | "Who struggled with subnetting?" | Chroma: semantic |
| `list_flagged()` | "What still needs my review?" | SQLite |

💡 The system prompt insists: *never guess a mark; numbers must come from the tools*. An LLM is bad at arithmetic over 150 rows, and pandas is perfect at it. Here's what `class_statistics` computes:

In [ ]:
from gradewise.reports.qa_agent import question_statistics

question_statistics(list_reports(exam.exam_id, db=STORE["db"])).head(6)

### 🎭 Rehearsal: a scripted conversation
A `ScriptedChatModel` plays Emma's assistant, calling each tool in turn; the tools and the stores are real.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

from gradewise.grading.fakes import ScriptedChatModel, tool_call_message
from gradewise.reports import build_report_agent

script = [
    tool_call_message("class_statistics", {}),
    tool_call_message("get_student_report", {"student_id": "student_05"}),
    tool_call_message("list_flagged", {}),
    AIMessage(
        "The hardest question is the top row of the statistics, student_05's marks are above, "
        "and one answer (student_10, Q8) still needs your review."
    ),
]
agent = build_report_agent(exam.exam_id, ScriptedChatModel(messages=iter(script)), **STORE)  # 🎭 stand-in model
state = agent.invoke({"messages": [{"role": "user", "content": "How did the class do, and what needs my attention?"}]})
for m in state["messages"][1:]:
    if isinstance(m, ToolMessage):
        print(f"🧰 {m.name}:\n   " + "\n   ".join(str(m.content).splitlines()[:4]) + "\n")
print("💬", state["messages"][-1].text)

🔑 Now for real. This cell stores the class again with **real embeddings** (in a separate sandbox folder: fake and real vectors must never share a collection), builds the agent with your chat model, and asks three questions with `ask()`. With `gpt-6-luna` and `text-embedding-3-small` that's well under a cent ($0 on `MODEL_MODE=free`).

In [ ]:
from gradewise.models import embeddings
from gradewise.reports import ask

QUESTIONS_FOR_EMMA = [
    "Which question did the class struggle with most?",
    "Why did student_05 lose marks on Q7a?",
    "Which answers still need my review, and why?",
]
try:
    REAL = {
        "db": SANDBOX / "real" / "gradewise.db",
        "embeddings": embeddings(),
        "persist_dir": SANDBOX / "real" / "vector_db",
    }
    for report in reports:
        store_report(rule_based_feedback(report, exam), exam, **REAL)
    real_agent = build_report_agent(exam.exam_id, **REAL)
    for q in QUESTIONS_FOR_EMMA:
        display(Markdown(f"**🧑‍🏫 {q}**\n\n{ask(q, exam.exam_id, agent=real_agent)}"))
except GradeWiseError as err:
    err.show()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">The report card is the part of GradeWise that parents, students and head teachers actually see, so it's where trust is won or lost. Showing the <b>criterion, the reason and the citation</b> behind every mark turns "the AI marked me down" into a conversation about the course notes. And "chat with results" turns a spreadsheet of marks into teaching insight in seconds: <i>"a third of the class lost the network-address mark on Q7a"</i> is a lesson plan for next week.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Today's code is <code>gradewise/reports/</code>: <code>writer.py</code> (<code>rule_based_feedback()</code>, <code>write_feedback()</code>, <code>render_html()</code>, <code>render_pdf()</code>, <code>save_report_files()</code>), <code>store.py</code> (<code>store_report()</code>, <code>report_documents()</code>, <code>search_reports()</code>) and <code>qa_agent.py</code> (<code>build_report_agent()</code>, <code>ask()</code>). Once you've graded real students on Day 4, chat with them from the terminal: <code>uv run week5/project/report_qa.py</code>. Tests: <code>uv run python -m pytest tests/test_reports.py -q</code>.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 📋 took apart a `ReportCard` and listed what makes feedback useful
- ✍️ wrote strengths and improvements with rules (free) and with an LLM (structured output)
- 🌐 rendered Sam's report card inline as HTML, proved student text is escaped, and drew the A4 PDF
- 💾 stored the class in SQLite and a Chroma `reports_cs_midterm` collection, with stable, upserting chunk IDs
- 💬 questioned the "chat with results" agent, whose numbers always come from tools

**Tomorrow (Day 4):** 🧱 the orchestrator. One LangGraph graph takes a pile of answer sheets all the way to stored report cards (transcribe, grade, review, write, store), **pausing for Emma** when something is flagged and resuming with her decision, even after a restart. Then we turn on **LangSmith** and watch a full run, call by call.

⏭ **Next:** `week5/day4.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>A class summary page for Emma.</b> Write <code>render_class_summary(reports)</code> that returns a small, self-contained HTML page: the class mean and grade-band counts at the top, then one row per question (from <code>question_statistics()</code>) with its mean mark as a bar, hardest first, and the flagged answers listed at the bottom. Escape every string! Show it inline with <code>IPython.display.HTML</code>. Bonus 🔑: build the report agent with <code>checkpointer=InMemorySaver()</code> and ask a follow-up in the same thread ("and which students got it right?"). Solution: <code>week5/solutions/day3_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• PyMuPDF Story (HTML → PDF): <a href="https://pymupdf.readthedocs.io/en/latest/recipes-stories.html">pymupdf.readthedocs.io/en/latest/recipes-stories.html</a><br/>• OWASP XSS prevention cheat sheet: <a href="https://cheatsheetseries.owasp.org/cheatsheets/Cross_Site_Scripting_Prevention_Cheat_Sheet.html">cheatsheetseries.owasp.org</a><br/>• Hattie & Timperley, <i>The Power of Feedback</i> (what makes feedback work)</span>
        </td>
    </tr>
</table>